# Smoke test: the capital of France

Check end to end that the four main-paper looped models work before reading any
aggregate figure. Each model loads through the same hooked TransformerLens bridge
as every other experiment, reads one short prompt with its native tokenizer, and
must do two things in a **single instrumented forward pass**:

1. predict a sensible next token and continue the text sensibly;
2. produce the attention stages of inference, measured as in
   [stages of inference](stages_of_inference.ipynb).

A per-recurrence readout then links the two. It shows at which recurrence the
prediction appears, and how that relates to the recurrent attention profile.
Each model's matched feedforward control runs alongside it.

Bridge logits are matched against the upstream implementations separately:
`tests/test_match_pretrained_models.py` covers the real Hugging Face checkpoints
at four recurrences, and `tests/test_match_parcae.py` a tiny Parcae export.
This notebook has no paper figures; outputs go only to its run directory.

In [ ]:
prompt = "The capital of France is"
expected = " Paris"  # A sensible next token, encoded by each model's own tokenizer.
recurrences = 32
ouro_recurrences = 4  # Ouro keeps four recurrences, as in stages of inference.
model_recurrences = None  # Defaults to the shared main group; optionally list pairs.
new_tokens = 8  # Length of the greedy continuation.
top_k = 5
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import gc
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D

from experiments.paper_models import PAPER_MODEL_GROUPS
from experiments.support import (
    BASELINE_FOR,
    COLORMAPS,
    MODEL_LABELS,
    _set_seed,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    configure_experiment,
    get_figsize,
    load_experiment_model,
    record_tokenization,
    save_figure,
    save_results,
)
from looped_internals.capture import compute_model_run_data, metrics_names_filter
from looped_internals.evaluation import coda_lens
from looped_internals.evaluation.logit_lens import logit_lens

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve the shared main group after parameter overrides; add matched controls.
if model_recurrences is None:
    model_recurrences = [
        (name, ouro_recurrences if name == "ouro_1_4b" else recurrences)
        for name in PAPER_MODEL_GROUPS["main"]
    ]
model_recurrences = [tuple(pair) for pair in model_recurrences]
control_pairs = list(
    dict.fromkeys((BASELINE_FOR[name], 1) for name, _ in model_recurrences)
)
configs = {
    (name, steps): configure_experiment(
        experiment="smoke_test",
        models=[name],
        num_steps=steps,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, steps in model_recurrences + control_pairs
}
config = next(iter(configs.values()))

In [ ]:
STAGE_METRICS = {
    "sink_rates": "Sink rate",
    "mixing_scores": "Mixing score",
    "colsum_concentrations": "Column concentration",
}


def next_token(logits, tokenizer, expected_id):
    """Top-k next tokens, and the expected token's probability."""
    probs = logits.float().softmax(-1)
    top = probs.topk(top_k)
    return {
        "top": [
            (tokenizer.decode([index]), p)
            for p, index in zip(top.values.tolist(), top.indices.tolist(), strict=True)
        ],
        "p_expected": probs[expected_id].item(),
        "expected_is_top": top.indices[0].item() == expected_id,
        "probs": probs,
    }


def head_mean(run, field):
    values = run.get_metric(field).float()
    return (values.mean(-1) if values.ndim > 1 else values).tolist()


@torch.no_grad()
def smoke_test(bridge, structure, steps):
    """Predict, measure attention stages, and read out each recurrence in one pass."""
    tokenizer = bridge.tokenizer
    tokens = tokenizer.encode(prompt, return_tensors="pt").to(config.device)
    expected_id = tokenizer.encode(expected, add_special_tokens=False)[0]
    # Huginn, the retrofits and Parcae draw a random initial latent state.
    _set_seed(seed)
    logits, cache = bridge.run_with_cache(
        tokens, accumulate=True, names_filter=metrics_names_filter, device="cpu"
    )
    _set_seed(seed)
    plain = bridge(tokens, return_type="logits")
    prediction = next_token(logits[0, -1], tokenizer, expected_id)
    run = compute_model_run_data(
        cache, tokens, structure, compute_pairwise=False, compute_differences=False
    )[0]

    # Decode the last recurrent layer after every recurrence, as the model would:
    # through its trained coda, if any, then the final norm and vocabulary head.
    layout = list(structure.virtual_layers(steps))
    core = [layer for layer in layout if layer.iteration is not None]
    has_coda = layout[-1].iteration is None
    readout = []
    for layer in core:
        if layer.layer_index == core[-1].layer_index:
            state = cache[layer.residual_key].to(config.device)
            lens = (
                coda_lens(bridge, state, input_ids=tokens)
                if has_coda
                else logit_lens(bridge, state)
            )
            readout.append(next_token(lens[0, -1], tokenizer, expected_id))
    del cache

    # Greedy continuation, reseeding so each pass draws the same initial state.
    ids = tokens
    for _ in range(new_tokens):
        _set_seed(seed)
        next_id = bridge(ids, return_type="logits")[0, -1].argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)

    return {
        "input_tokens": [tokenizer.decode([index]) for index in tokens[0].tolist()],
        "prediction": {k: v for k, v in prediction.items() if k != "probs"},
        "continuation": tokenizer.decode(ids[0, tokens.shape[1] :].tolist()),
        "readout": [{k: v for k, v in r.items() if k != "probs"} for r in readout],
        # Attention capture must not change the logits.
        "hooked_vs_plain": (logits - plain).abs().max().item(),
        # The final recurrence's readout must reproduce the model's own prediction.
        "readout_vs_output": (
            (readout[-1]["probs"] - prediction["probs"]).abs().max().item()
            if readout
            else None
        ),
        "layers": [block.layer_index for block in run.per_virtual_layer_metrics],
        "recurrent_layers": run.recurrent_layer_indices,
        "curves": {field: head_mean(run, field) for field in STAGE_METRICS},
    }

## Generate and reduce

The shared main group supplies Ouro at four recurrences and Huginn, Retrofitted
Llama and Parcae 770M at 32, as in stages of inference. Llama 3.2 1B (the control
for Ouro, Huginn and Retrofitted Llama) and the trained one-recurrence Parcae 770M
control each run once. Every model uses its native tokenization: Huginn and the
Llama-based models prepend BOS, while Ouro and Parcae do not.

Each model makes one hooked forward pass, caching block outputs and attention
patterns, and one ordinary pass to check that capture leaves the logits unchanged.
Both reseed first, because Huginn, the retrofits and Parcae initialise their latent
state randomly. The greedy continuation reruns the full prefix for each new token,
without a KV cache. Every model is released before the next loads. `overview.pt`
keeps the predictions, readouts, layer identities and head-averaged curves.

In [ ]:
settings = asdict(config)
for field in (
    "models",
    "num_steps",
    "dataset",
    "task",
    "limit",
    "include_baselines",
    "compute_residual_entropy",
):
    del settings[field]
settings.update(
    output_dir=str(config.output_dir),
    prompt=prompt,
    expected=expected,
    model_recurrences=model_recurrences,
    control_pairs=control_pairs,
    new_tokens=new_tokens,
    top_k=top_k,
    readout="last_recurrent_layer_then_coda_if_any_then_final_norm_and_head",
    versions={
        name: version(name) for name in ("torch", "transformer-lens", "transformers")
    },
)
run_dir = save_results({}, settings, config.output_dir)

results = {}
for pair, pair_config in configs.items():
    name, steps = pair
    _set_seed(seed)
    bridge, structure = load_experiment_model(name, pair_config)
    results[pair] = smoke_test(bridge, structure, steps)
    record_tokenization(run_dir, name, bridge)
    del bridge
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
torch.save(results, run_dir / "overview.pt")

## Next-token predictions

A tick marks a model whose most probable next token is the expected one. Input
tokens are shown as the model receives them, including any native BOS. Each model
also passes two internal checks. Hooked and ordinary logits must agree, and the
final recurrence's readout must reproduce the model's own next-token distribution.

In [ ]:
for pair, result in results.items():
    name, steps = pair
    prediction = result["prediction"]
    passes = "one pass" if pair in control_pairs else f"{steps} recurrences"
    print(
        f"{'✓' if prediction['expected_is_top'] else '✗'} {MODEL_LABELS[name]}, {passes}"
    )
    print("    input:  " + " | ".join(map(repr, result["input_tokens"])))
    print("    next:   " + "  ".join(f"{t!r} {p:.3f}" for t, p in prediction["top"]))
    print(f"    greedy: {result['continuation']!r}")
    checks = f"max |Δ logit| hooked vs plain = {result['hooked_vs_plain']:.1e}"
    if result["readout_vs_output"] is not None:
        checks += (
            f"; max |Δp| final readout vs output = {result['readout_vs_output']:.1e}"
        )
    print(f"    checks: {checks}")

## What the figure shows

Columns are models; the first three rows repeat the attention measurements from
[stages of inference](stages_of_inference.ipynb), and the last row is the readout.

| Row | Interpretation |
| --- | --- |
| Sink rate | Fraction of heads whose mean attention to the first token is at least 0.3. |
| Mixing score | Attention-row entropy divided by its causal maximum, averaged over query rows excluding the first, then over heads. |
| Column concentration | One minus the normalised entropy of attention received across key positions, averaged over heads. |
| Readout | Probability of the expected token at the final position after each recurrence. The last recurrent layer's output is read through the trained coda (Huginn, Retrofitted Llama, Parcae), then the final norm and head. Ouro has no coda; its final norm and head give its native per-recurrence exit readout. Filled circles mark recurrences where the expected token is most probable; crosses mark those where it is not. |

Layer depth divides each recorded layer index by its maximum: prelude, core and
coda keep their positions, and every recurrence overlays the same core span.
Colours run from early (green) to late (yellow) recurrences, as in the paper.
Prelude is orange, coda black, and the matched feedforward control dashed red,
on its own 0–100% axis. In the readout row, the control's dashed line is its
probability for the same token. Faint grey links join the prelude to the first
recurrence, and the final recurrence to the coda.

This is a **single short sequence**, so no uncertainty bands are shown. With five
or six tokens, attention entropies take few distinct values. The profiles should
resemble, not reproduce, the dataset averages in stages of inference.

In [ ]:
def depth(layers):
    layers = np.asarray(layers)
    return layers / max(layers.max(), 1) * 100


def recurrence_groups(result):
    """Use recorded layer identities to separate prelude, recurrences, and coda."""
    layers = np.asarray(result["layers"])
    core = result["recurrent_layers"]
    groups = {}
    if np.any(layers < min(core)):
        groups["Prelude"] = np.flatnonzero(layers < min(core))
    by_layer = np.asarray([np.flatnonzero(layers == layer) for layer in core])
    for recurrence, indices in enumerate(by_layer.T):
        groups[f"R{recurrence}"] = indices
    if np.any(layers > max(core)):
        groups["Coda"] = np.flatnonzero(layers > max(core))
    return groups


def draw_stages(ax, pair, field):
    result = results[pair]
    x, y = depth(result["layers"]), np.asarray(result["curves"][field])
    groups = recurrence_groups(result)
    n_recurrences = sum(key.startswith("R") for key in groups)
    cmap = plt.colormaps[COLORMAPS["recurrences"]]
    lines = {}
    for key, indices in groups.items():
        if key in {"Prelude", "Coda"}:
            color = "orange" if key == "Prelude" else "black"
        else:
            color = cmap(int(key[1:]) / max(n_recurrences - 1, 1))
        (lines[key],) = ax.plot(
            x[indices],
            y[indices],
            color=color,
            marker="." if len(indices) == 1 else None,
            linewidth=1.5,
        )
    add_prelude_coda_connectors(ax, lines)
    control = results[(BASELINE_FOR[pair[0]], 1)]
    ax.plot(
        depth(control["layers"]),
        control["curves"][field],
        color="red",
        linestyle="--",
        linewidth=1.5,
    )


def draw_readout(ax, pair):
    readout = results[pair]["readout"]
    x = np.arange(1, len(readout) + 1)
    p = np.asarray([r["p_expected"] for r in readout])
    top = np.asarray([r["expected_is_top"] for r in readout])
    colors = plt.colormaps[COLORMAPS["recurrences"]](np.linspace(0, 1, len(readout)))
    ax.plot(x, p, color="grey", linewidth=0.8, zorder=1)
    ax.scatter(x[top], p[top], color=colors[top], marker="o", s=12, zorder=2)
    ax.scatter(x[~top], p[~top], color=colors[~top], marker="x", s=12, zorder=2)
    control = results[(BASELINE_FOR[pair[0]], 1)]["prediction"]["p_expected"]
    ax.axhline(control, color="red", linestyle="--", linewidth=1.5)
    ax.set(xlabel="Recurrence", ylim=(0, 1))


def plot_smoke_test(pairs, filename):
    rows = len(STAGE_METRICS) + 1
    fig, axes = plt.subplots(
        rows,
        len(pairs),
        squeeze=False,
        figsize=get_figsize(True, (rows, len(pairs)), 1.3),
        layout="constrained",
    )
    for column, pair in enumerate(pairs):
        axes[0, column].set_title(MODEL_LABELS[pair[0]])
        for row, field in enumerate(STAGE_METRICS):
            ax = axes[row, column]
            draw_stages(ax, pair, field)
            if field == "sink_rates":
                ax.set_ylim(0, 1.05)  # Saturates on short inputs; keep the full scale.
            if row:
                ax.sharex(axes[0, column])
            ax.tick_params(labelbottom=row == len(STAGE_METRICS) - 1)
        axes[len(STAGE_METRICS) - 1, column].set_xlabel("Layer depth (%)")
        draw_readout(axes[-1, column], pair)
    for row, label in enumerate(STAGE_METRICS.values()):
        axes[row, 0].set_ylabel(label)
    axes[-1, 0].set_ylabel(f"P({expected.strip()})")
    handles = [
        Line2D([], [], color="orange", linewidth=2, label="Prelude"),
        Line2D([], [], color="black", linewidth=2, label="Coda"),
        Line2D([], [], color="red", linestyle="--", linewidth=2, label="Control"),
        Line2D([], [], color="grey", marker="o", linestyle="", label="Top token"),
        Line2D([], [], color="grey", marker="x", linestyle="", label="Not top"),
    ]
    fig.legend(handles=handles, loc="outside upper center", ncols=len(handles))
    add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Stages of inference and the emerging prediction

Read each column from top to bottom: the attention profile of every recurrence,
then what that recurrence predicts.

In [ ]:
plot_smoke_test(model_recurrences, "smoke_test")
print(run_dir)

## Reading the result

The smoke test passes when every model ranks the expected token first, continues
the text sensibly, and reports zero (or float-rounding) differences in both
checks. With the default prompt and seed, all four looped models and both controls
rank " Paris" first and continue with a sentence about Paris. In float32 on CPU,
both checks are exactly zero.

The figure then links the prediction to the stages of inference:

- **The attention profile changes most in the first few recurrences, and so does
  the prediction.** Each recurrence's profile approaches the final recurrence's
  within a handful of steps; from then on, later recurrences overlay one another.
  Over those same early recurrences the readout rises and then levels off. Ouro
  goes from an almost uniform distribution after one recurrence to its final
  prediction after three, as its profile reaches its final shape. Retrofitted
  Llama and Parcae settle within roughly five recurrences.
- **Huginn takes longest to form the answer.** " Paris" becomes its top token only
  at the fourth recurrence, peaks around the fifth to eighth, and then declines
  slowly while staying first, although its attention profile is already stable.
  A settled stage profile accompanies the prediction but does not fix its
  confidence.
- **Prelude and coda play the early and late stages.** Retrofitted Llama's
  prelude and coda inherit Llama 3.2 1B's first and last layers, and follow the
  control's first and last stages closely; the recurrent core repeats a
  middle-stage profile.

This is one prompt, so treat it as a sanity check rather than evidence. The
dataset-averaged figures in stages of inference carry the paper's claims; this
notebook shows that the same instrumented forward pass also produces sensible
text. Values may differ slightly in bfloat16 on GPU.